<a href="https://colab.research.google.com/github/nabilah-afrin/recommendation_system_rokomri_books/blob/secondary/notebooks/preprocessing_bn_books.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# %cd /content/drive/MyDrive/Rokomari Recommendation Dataset
%cd /content/drive/MyDrive/Dataset/rokomari_books/Rokomari Recommendation Dataset/Datasets

/content/drive/.shortcut-targets-by-id/1SdeIcOv6xY-c8y2UcMwPLYx_BaB0zdXx/Rokomari Recommendation Dataset/Datasets


In [ ]:
!ls

 corrected_language.csv        rokomari_book_data_v2.csv	  'scraping log.txt'
'Data Analysis Report.gdoc'    rokomari_books_only_bangla_v2.csv   wrong_language_url.txt
 mixed_title_rokomari_v2.csv   rokomari_v2.csv
 rokomari_book_data.csv        rokomari_v2.ipynb


In [ ]:
# !pip install googletrans==4.0.0-rc1

In [ ]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 42.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993222 sha256=f9df40f25694016803faf06982ffe30cf2b37f0155b250b700f299f9067fa07a
  Stored in directory: /root/.cache/pip/wheels/95/03/7d/59ea870c70ce4e5a370638b5462a7711ab78fba2f655d05106
Successfully built langdetect


In [ ]:
!pip install deep_translator --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import langdetect as ld
# import googletrans as gt
import deep_translator as dt
from deep_translator import GoogleTranslator

# Only Bangla Books

In [ ]:
# load the rokomari_bn_books.csv
# uncomment when loading from your drive
# df_bn = pd.read_csv("/content/drive/MyDrive/Rokomari Recommendation Dataset/df_bn.csv")

df_bn = pd.read_csv("rokomari_books_only_bangla_v2.csv")

In [ ]:
df_bn.head(5)

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,offer_price,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title,language_categories,categories_fixed
0,350167,অমানুষিক,মনোরঞ্জন ব্যাপারী,একা,Eka (India),পশ্চিমবঙ্গের বই,West Bengal Books,6 March 2023,9789357762298,No summary,...,450.0,https://www.rokomari.com/book/350167/amanushik,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,অমানুষিক,bn,bn,পশ্চিমবঙ্গের বই
1,377700,নির্বাচিত গল্প সংকলন,লু স্যুন,ছাড়পত্র প্রকাশন (ইন্ডিয়া),Charpatra Prakashan (India),পশ্চিমবঙ্গের বই: সমকালীন গল্প,West Bengal Books: Contemporary Story,Edition,9788194097303,No summary,...,320.0,https://www.rokomari.com/book/377700/nirbachit...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,নির্বাচিত গল্প সংকলন,bn,bn,"পশ্চিমবঙ্গের বই, সমকালীন গল্প"
2,377718,তিমুর ও তার দলবল,আর্কাদি গাইদার,ছাড়পত্র প্রকাশন (ইন্ডিয়া),Charpatra Prakashan (India),পশ্চিমবঙ্গের বই: শিশু-কিশোর উপন্যাস,West Bengal Books: Children and Teens Novel,Edition,9788193860939,No summary,...,180.0,https://www.rokomari.com/book/377718/timur-o-t...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,তিমুর ও তার দলবল,bn,bn,"পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস"
3,189529,চেক ডিসঅনার মামলার সহজ ভাষ্য,মোঃ কাইছার হামিদ,এ.কে লিগ্যাল সল্যুশন,A.K Legal Solution,ব্যাংকিং এন্ড কমার্স ল,Banking and Commerce Law,1st Published,No ISBN,* চেক ডিসঅনার ও মামলা দায়েরের পদ্ধতি সংক্রান্...,...,175.0,https://www.rokomari.com/book/189529/cheque-di...,https://img.cf.rokomari.com/ProductNew20190903...,Not Available,book,5.0,চেক ডিসঅনার মামলার সহজ ভাষ্য,bn,bn,ব্যাংকিং এন্ড কমার্স ল
4,325781,হাইকোর্ট এক্সাম ফর্মুলা,মোঃ কাইছার হামিদ,A.K Legal Solution,A.K Legal Solution,অ্যাডভোকেসি/বিচার আইন,Advocacy/ Adjudication Law,Edition,9789843545589,"""High Court Exam Formula"" book will be very he...",...,500.0,https://www.rokomari.com/book/325781/high-cour...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,হাইকোর্ট এক্সাম ফর্মুলা,bn,bn,"অ্যাডভোকেসি, বিচার আইন"


In [ ]:
def detect_language(row):
    try:
        lang = ld.detect(str(row))
    except:
        return 'unknown'
    return lang

In [ ]:
df_bn['language_author'] = df_bn['author'].apply(detect_language)

In [ ]:
df_bn.head()

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title,language_categories,categories_fixed,language_author
0,350167,অমানুষিক,মনোরঞ্জন ব্যাপারী,একা,Eka (India),পশ্চিমবঙ্গের বই,West Bengal Books,6 March 2023,9789357762298,No summary,...,https://www.rokomari.com/book/350167/amanushik,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,অমানুষিক,bn,bn,পশ্চিমবঙ্গের বই,bn
1,377700,নির্বাচিত গল্প সংকলন,লু স্যুন,ছাড়পত্র প্রকাশন (ইন্ডিয়া),Charpatra Prakashan (India),পশ্চিমবঙ্গের বই: সমকালীন গল্প,West Bengal Books: Contemporary Story,Edition,9788194097303,No summary,...,https://www.rokomari.com/book/377700/nirbachit...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,নির্বাচিত গল্প সংকলন,bn,bn,"পশ্চিমবঙ্গের বই, সমকালীন গল্প",bn
2,377718,তিমুর ও তার দলবল,আর্কাদি গাইদার,ছাড়পত্র প্রকাশন (ইন্ডিয়া),Charpatra Prakashan (India),পশ্চিমবঙ্গের বই: শিশু-কিশোর উপন্যাস,West Bengal Books: Children and Teens Novel,Edition,9788193860939,No summary,...,https://www.rokomari.com/book/377718/timur-o-t...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,তিমুর ও তার দলবল,bn,bn,"পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস",bn
3,189529,চেক ডিসঅনার মামলার সহজ ভাষ্য,মোঃ কাইছার হামিদ,এ.কে লিগ্যাল সল্যুশন,A.K Legal Solution,ব্যাংকিং এন্ড কমার্স ল,Banking and Commerce Law,1st Published,No ISBN,* চেক ডিসঅনার ও মামলা দায়েরের পদ্ধতি সংক্রান্...,...,https://www.rokomari.com/book/189529/cheque-di...,https://img.cf.rokomari.com/ProductNew20190903...,Not Available,book,5.0,চেক ডিসঅনার মামলার সহজ ভাষ্য,bn,bn,ব্যাংকিং এন্ড কমার্স ল,bn
4,325781,হাইকোর্ট এক্সাম ফর্মুলা,মোঃ কাইছার হামিদ,A.K Legal Solution,A.K Legal Solution,অ্যাডভোকেসি/বিচার আইন,Advocacy/ Adjudication Law,Edition,9789843545589,"""High Court Exam Formula"" book will be very he...",...,https://www.rokomari.com/book/325781/high-cour...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,হাইকোর্ট এক্সাম ফর্মুলা,bn,bn,"অ্যাডভোকেসি, বিচার আইন",bn


In [ ]:
df_bn['language_author'].value_counts()

,count
language_author,
bn,204248
ar,296
en,11
id,7
af,5
fr,1
tr,1
sv,1
so,1


# Correcting author names

In [ ]:
# There are no rows with 'No Author' "

df_bn.loc[df_bn['author'] == 'No Author'].shape

(0, 26)

In [ ]:
# df_bn = df_bn.loc[~(df_bn['author'] == 'No Author')]

In [ ]:
# Deletion successful

# df_bn.loc[df_bn['author'] == 'No Author'].shape

In [ ]:
translator = GoogleTranslator(source='auto', target='bn')
text = "Saifur Rahman Khan"
translation = translator.translate(text=text)
translation

'সাইফুর রহমান খান'

In [ ]:
import re

def correct_author_names(row):
    """
    The unicode range between \u0980 - \u09FF defines the Bangla characters
    and digits in the Unicode character set
    """
    return re.sub(r'[^\u0980-\u09FF ]+', '', str(row)).strip()

def translate_author_names(row):
    text = str(row)
    translator = gt.Translator()
    translation = translator.translate(text=text, dest='bn')

    return translation.text

def translate_to_bangla(row):
    return GoogleTranslator(source='auto', target='bn').translate(text=str(row))

def translate_from_id(row):
    return GoogleTranslator(source='en', target='bn').translate(text=str(row))




In [ ]:
for language in df_bn['language_author'].value_counts().index:
    if language == 'bn' or language == 'ar':
        continue
    else:
        print(f"Correcting Language: {language}")
        filt = (df_bn['language_author'] == language, 'author')
        df_bn.loc[filt] = df_bn.loc[filt].apply(translate_to_bangla)

print("\nDone")

Correcting Language: en
Correcting Language: id
Correcting Language: af
Correcting Language: fr
Correcting Language: tr
Correcting Language: sv
Correcting Language: so
Correcting Language: nl

Done


## Let's see the value counts of the author_languages again

In [ ]:
df_bn['language_author'] = df_bn['author'].apply(detect_language)

In [ ]:
df_bn['language_author'].value_counts()

,count
language_author,
bn,204274
ar,288
en,6
id,3
tr,1


In [ ]:
df_bn.loc[df_bn['language_author'] == 'en']

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title,language_categories,categories_fixed,language_author
20731,228238,"ডিজিটাল ব্যাংক, নিও ব্যাংক : ভবিষ্যৎ ব্যাংকিং","ড. জাহিদুজ্জামান, CIPA, CSAA, CIFE ( Certified...",তাম্রলিপি,Tamrolipi,ব্যাংকিং ও ফিন্যান্স,Banking and Finance,1st Edition,No ISBN,সামনের দুনিয়া প্রযুক্তিময় দুনিয়া। প্রযুক্তি...,...,https://www.rokomari.com/book/228238/digital-b...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,5.0,ডিজিটাল ব্যাংক নিও ব্যাংক : ভবিষ্যৎ ব্যাংকিং,bn,bn,"ব্যাংকিং, ফিন্যান্স",en
147551,284249,ইসলামী ব্যাংকিং এবং ফাইন্যান্সঃ মূলনীতি ও অনুশীলন,"ড. জাহিদুজ্জামান, CIPA, CSAA, CIFE ( Certified...",ব্রেইনারি পাবলিকেশন,Brainery Publication,ইসলামি অর্থনীতি ও ব্যবসা বাণিজ্য,Islamic Financial and Business,1st Edition 2023,978-984-96702-4-7,রাসুল (সা:) এর যুগ থেকে শুরু করে সাহাবায়ে কেরা...,...,https://www.rokomari.com/book/284249/islamic-b...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,ইসলামী ব্যাংকিং এবং ফাইন্যান্সঃ মূলনীতি ও অনুশীলন,bn,bn,"ইসলামি অর্থনীতি, ব্যবসা বাণিজ্য",en
147552,283901,ইসলামিক ফিনটেক,"ড. জাহিদুজ্জামান, CIPA, CSAA, CIFE ( Certified...",ব্রেইনারি পাবলিকেশন,Brainery Publication,ইসলামি অর্থনীতি ও ব্যবসা বাণিজ্য,Islamic Financial and Business,1st Published,9789849670223,মুসলিম জনগণ ডিজিটাল ফিন্যান্স এবং শারিয়াহ নীত...,...,https://www.rokomari.com/book/283901/islamic-f...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0,ইসলামিক ফিনটেক,bn,bn,"ইসলামি অর্থনীতি, ব্যবসা বাণিজ্য",en
147553,283891,ফিনটেক,"ড. জাহিদুজ্জামান, CIPA, CSAA, CIFE ( Certified...",ব্রেইনারি পাবলিকেশন,Brainery Publication,ব্যাংকিং ও ফিন্যান্স,Banking and Finance,3rd Published,9789849670230,আর্থিক (ফাইন্যান্সিয়াল) খাতে প্রযুক্তির (টেকনো...,...,https://www.rokomari.com/book/283891/fintech,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0,ফিনটেক,bn,bn,"ব্যাংকিং, ফিন্যান্স",en
163435,113205,স্বপ্ন সুন্দর ভালবাসা,Josim Chowdhury(জসিম চৌধুরী)-12455,বলাকা প্রকাশন (চট্টগ্রাম),Balaka Prokashon (Chittagong),বাংলা কবিতা,Poem- Bangla,1st Published,9789849161752,No summary,...,https://www.rokomari.com/book/113205/swapno-su...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0,স্বপ্ন সুন্দর ভালবাসা,bn,bn,বাংলা কবিতা,en
203731,339170,"অটিজম বাংলাদেশ প্রেক্ষিত, সাফল্য ও সম্ভাবনা",Josim Chowdhury(জসিম চৌধুরী)-12455,ঝুমঝুমি প্রকাশন,Zhumzhumi prokashan,শারীরিক ও মানসিক প্রতিবন্ধকতা,Physical and Mental Disorder,1st Edition,9789849748199,লেখক জসিম চৌধুরী তার গ্রন্থে যেমন বৈশ্বিক অটিজ...,...,https://www.rokomari.com/book/339170/autism-ba...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0,অটিজম বাংলাদেশ প্রেক্ষিত সাফল্য ও সম্ভাবনা,bn,bn,"শারীরিক, মানসিক প্রতিবন্ধকতা",en


In [ ]:
df_bn.loc[df_bn['language_author'] == 'tr']

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title,language_categories,categories_fixed,language_author
44759,408265,বহুমাত্রিক রবীন্দ্রনাথ,Prabir Kumar Pal,রিভার্স পাবলিকেশন,Reverse Publication,পশ্চিমবঙ্গের বই: প্রবন্ধ,West Bengal Books: Articles,Edition,9789392283093,No summary,...,https://www.rokomari.com/book/408265/bohumatri...,https://img.cf.rokomari.com/ProductNew20190903...,Request for Stock,book,0.0,বহুমাত্রিক রবীন্দ্রনাথ,bn,bn,"পশ্চিমবঙ্গের বই, প্রবন্ধ",tr


In [ ]:
df_bn.loc[df_bn['language_author'] == 'id']

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title,language_categories,categories_fixed,language_author
163,417530,ইশ্বর আসছেন,Joydeep Lahiri,অক্ষর সংলাপ প্রকাশন,Akshar Sanglap Prakashan (India),পশ্চিমবঙ্গের বই,West Bengal Books,No Edition,No ISBN,No summary,...,https://www.rokomari.com/book/417530/iswar-aschen,https://img.cf.rokomari.com/ProductNew20190903...,Request for Stock,book,0.0,ইশ্বর আসছেন,bn,bn,পশ্চিমবঙ্গের বই,id
54633,283863,স্থান কাল মহাকর্ষ ও অপেক্ষবাদ,Biswaranjan Nag,পশ্চিমবঙ্গ রাজ্য পুস্তক পর্ষৎ (ভারত),Poshcimbongo Rajjo Pustok Porshot (India),পশ্চিমবঙ্গের বই,West Bengal Books,1st published,8124702535,No summary,...,https://www.rokomari.com/book/283863/sthan-kal...,https://img.cf.rokomari.com/ProductNew20190903...,Request for Stock,book,0.0,স্থান কাল মহাকর্ষ ও অপেক্ষবাদ,bn,bn,পশ্চিমবঙ্গের বই,id
200740,415552,মণিমালা,Prasanta Kumar Bhowmik,ঋক পাব্লিকেশন,Rick Publications,পশ্চিমবঙ্গের বই,West Bengal Books,Edition,978819578212,No summary,...,https://www.rokomari.com/book/415552/monimala,https://img.cf.rokomari.com/ProductNew20190903...,Request for Stock,book,0.0,মণিমালা,bn,bn,পশ্চিমবঙ্গের বই,id


In [ ]:
# changing manually the author name Prabir Kumar Pal (সম্পাদক) to প্রবীর কুমার পাল

df_bn.loc[df_bn['author'] == 'Prabir Kumar Pal', 'author'] = 'প্রবীর কুমার পাল'

In [ ]:
df_bn['language_author'] = df_bn['author'].apply(detect_language)

In [ ]:
df_bn['language_author'].value_counts()

,count
language_author,
bn,204278
ar,287
en,4
id,3


In [ ]:
for language in df_bn['language_author'].value_counts().index:
    if language == 'bn' or language == 'ar':
        continue
    else:
        print(f"Correcting Language: {language}")
        filt = (df_bn['language_author'] == language, 'author')
        df_bn.loc[filt] = df_bn.loc[filt].apply(translate_to_bangla)

print("\nDone")

Correcting Language: en
Correcting Language: id

Done


In [ ]:
df_bn.loc[df_bn['language_author'] == 'id', 'author'] = df_bn.loc[df_bn['language_author'] == 'id', 'author'].apply(translate_from_id)

In [ ]:
df_bn['language_author'] = df_bn['author'].apply(detect_language)
df_bn['language_author'].value_counts()

,count
language_author,
bn,204266
ar,301
en,5


In [ ]:
# let's see the names with arabic ones

for idx, names in enumerate(df_bn.loc[df_bn['language_author'] == 'ar', 'author']):
    print(f"{idx} | {names}")

0 | حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)
1 | حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)
2 | حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)
3 | القاضي ناصر الدين عبد الله بن عمر البيضاوي (কাজী নাসিরুদ্দিন আব্দুল্লাহ বিন আমর বায়যাবী)
4 | حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)
5 | حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)
6 | حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)
7 | حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)
8 | حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)
9 | حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)
10 | حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)
11 | حكيم الامت مولانا اشرف علي تهانوي رح ( হা

In [ ]:
# changing this name حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.) (সম্পাদক) to ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)

df_bn.loc[df_bn['author'] == 'حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.) (সম্পাদক)', 'author'] = "হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ."

In [ ]:
# no more
df_bn.loc[df_bn['author'] == 'حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.) (সম্পাদক)', 'author'].shape

(0,)

## Fixing the mixed language in author names (Arabic and Bangla)

In [ ]:
df_bn.loc[df_bn['language_author'] == 'ar', 'author']

,author
1212,حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল...
5222,حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল...
5224,حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল...
5537,القاضي ناصر الدين عبد الله بن عمر البيضاوي (কা...
5563,حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল...
...,...
191114,(شيخ الاسلام مفتي محمد تقي عثماني) শাইখুল ইসলা...
191571,حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল...
191576,حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল...
194455,حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল...


In [ ]:
filt = (df_bn['language_author'] == 'ar', 'author')
df_bn.loc[filt] = df_bn.loc[filt].apply(correct_author_names)

In [ ]:
for idx, name in enumerate(df_bn.loc[filt]):
    print(f"{idx} | {name}")

0 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
1 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
2 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
3 | কাজী নাসিরুদ্দিন আব্দুল্লাহ বিন আমর বায়যাবী
4 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
5 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
6 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
7 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
8 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
9 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
10 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
11 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
12 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
13 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
14 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
15 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
16 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
17 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
18 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
19 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
20 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ
21 | হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভ

## Fixing the mixed language in author names (English and Bangla)

In [ ]:
filt = (df_bn['language_author'] == 'en', 'author')

In [ ]:
df_bn.loc[filt] = df_bn.loc[filt].apply(correct_author_names)

In [ ]:
for idx, name in enumerate(df_bn.loc[filt]):
    print(f"{idx} | {name}")

0 | ড জাহিদুজ্জামান
1 | ড জাহিদুজ্জামান
2 | ড জাহিদুজ্জামান
3 | ড জাহিদুজ্জামান
4 | জসিম চৌধুরী


In [ ]:
df_bn['language_author'] = df_bn['author'].apply(detect_language)

In [ ]:
df_bn['language_author'].value_counts()

,count
language_author,
bn,204423
ar,149


In [ ]:
df_bn.loc[df_bn['language_author'] == 'ar', 'author'].values

array(['حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)',
       'حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)',
       'حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)',
       'حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)',
       'حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)',
       'حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)',
       'حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)',
       'حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)',
       'حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)',
       'حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থানভী রহ.)',
       'حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল উম্মত মাওলানা আশরাফ আলী থ

 So there were other mixed language values that the language detect couldn't detect at first time

## Fixing all author names (keeping only with bangla letters)

In [ ]:
df_bn.loc[:, 'author'] = df_bn['author'].apply(correct_author_names)

In [ ]:
df_bn['language_author'] = df_bn['author'].apply(detect_language)

In [ ]:
df_bn['language_author'].value_counts()

,count
language_author,
bn,204572


All Bangla! yay!

In [ ]:
df_bn.to_csv("rokomari_books_only_bangla_v2.csv", index=False)